In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Mag-transpile gamit ang mga pass manager

<details>
<summary><b>Mga bersyon ng package</b></summary>

Ang code sa pahinang ito ay ginawa gamit ang mga sumusunod na kinakailangan.
Inirerekomenda naming gamitin ang mga bersyong ito o mas bago.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Ang inirerekomendang paraan ng pag-transpile ng circuit ay ang lumikha ng staged pass manager at pagkatapos ay i-execute ang `run` method nito kasama ang circuit bilang input. Ipinapaliwanag ng pahinang ito kung paano mag-transpile ng mga quantum circuit sa ganitong paraan.
## Ano ang isang (staged) pass manager?
Sa konteksto ng Qiskit SDK, ang transpilation ay tumutukoy sa proseso ng pag-transform ng isang input circuit sa isang anyo na angkop para sa pagpapatakbo sa isang quantum device. Karaniwang nangyayari ang transpilation sa isang pagkakasunud-sunod ng mga hakbang na tinatawag na transpiler passes. Pinoproseso ng circuit ang bawat transpiler pass nang sunud-sunod, kung saan ang output ng isang pass ang nagiging input sa susunod. Halimbawa, maaaring dumaan ang isang pass sa circuit at pagsamahin ang lahat ng magkakasunod na sequence ng single-qubit gate, at pagkatapos ay maaaring i-synthesize ng susunod na pass ang mga gate na ito sa basis set ng target na device. Ang mga transpiler pass na kasama sa Qiskit ay matatagpuan sa [qiskit.transpiler.passes](https://docs.quantum.ibm.com/api/qiskit/transpiler_passes) module.

Ang pass manager ay isang object na nag-iimbak ng listahan ng mga transpiler pass at maaaring i-execute ang mga ito sa isang circuit. Gumawa ng pass manager sa pamamagitan ng pag-initialize ng [`PassManager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager) na may listahan ng mga transpiler pass. Para patakbuhin ang transpilation sa isang circuit, tawagan ang [`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager#run) method na may circuit bilang input.

Ang staged pass manager ay isang espesyal na uri ng pass manager na kumakatawan sa isang antas ng abstraction na mas mataas kaysa sa isang normal na pass manager. Habang ang isang normal na pass manager ay binubuo ng ilang transpiler pass, ang isang staged pass manager ay binubuo ng ilang *pass manager*. Ito ay isang kapaki-pakinabang na abstraction dahil karaniwang nangyayari ang transpilation sa mga discrete na yugto, tulad ng inilarawan sa [Transpiler stages](transpiler-stages), kung saan ang bawat yugto ay kinakatawan ng isang pass manager. Ang mga staged pass manager ay kinakatawan ng [`StagedPassManager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.StagedPassManager) na klase. Inilalarawan ng natitirang bahagi ng pahinang ito kung paano lumikha at mag-customize ng mga (staged) pass manager.
## Gumawa ng preset staged pass manager
Para lumikha ng preset staged pass manager na may makatwirang mga default, gamitin ang [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) function:

In [1]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.backend("ibm_fez")
pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

Para mag-transpile ng isang circuit o listahan ng mga circuit gamit ang pass manager, ipasa ang circuit o listahan ng mga circuit sa `run` method. Gawin natin ito sa isang two-qubit circuit na binubuo ng isang Hadamard na sinusundan ng dalawang magkadikit na CX gate:

In [2]:
from qiskit import QuantumRegister, QuantumCircuit

# Create a circuit
qubits = QuantumRegister(2, name="q")
circuit = QuantumCircuit(qubits)
a, b = qubits
circuit.h(a)
circuit.cx(a, b)
circuit.cx(b, a)

# Transpile it by calling the run method of the pass manager
transpiled = pass_manager.run(circuit)

# Draw it, excluding idle qubits from the diagram
transpiled.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/dcc69b72-e13b-4df6-a51f-a5ef2108bae7-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/dcc69b72-e13b-4df6-a51f-a5ef2108bae7-0.svg)

Tingnan ang [Transpilation defaults and configuration options](defaults-and-configuration-options) para sa paglalarawan ng mga posibleng argumento sa `generate_preset_pass_manager` function. Ang mga argumento sa `generate_preset_pass_manager` ay tumutugma sa mga argumento sa [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile) function.

<CodeAssistantAdmonition tagLine="Nahihirapan kang matandaan ang mga detalye ng pass manager? Subukang tanungin ang Qiskit Code Assistant." />

Kung hindi natutugunan ng mga preset pass manager ang iyong mga pangangailangan, i-customize ang transpilation sa pamamagitan ng paggawa ng (staged) pass manager o maging mga transpilation pass. Inilalarawan ng natitirang bahagi ng pahinang ito kung paano lumikha ng mga pass manager. Para sa mga tagubilin kung paano lumikha ng mga transpilation pass, tingnan ang [Write your own transpiler pass](custom-transpiler-pass).

## Gumawa ng sariling pass manager

Ang [qiskit.transpiler.passes](https://docs.quantum.ibm.com/api/qiskit/transpiler_passes) module ay naglalaman ng maraming transpiler pass na maaaring gamitin para gumawa ng mga pass manager. Para gumawa ng pass manager, i-initialize ang isang `PassManager` na may listahan ng mga pass. Halimbawa, ang sumusunod na code ay gumagawa ng transpiler pass na pinagsasama ang mga magkadikit na two-qubit gate at pagkatapos ay sina-synthesize ang mga ito sa isang basis ng [$R_y$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RYGate), [$R_z$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RZGate), at [$R_{xx}$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RXXGate) gate.

In [3]:
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import (
    Collect2qBlocks,
    ConsolidateBlocks,
    UnitarySynthesis,
)

basis_gates = ["rx", "ry", "rxx"]
translate = PassManager(
    [
        Collect2qBlocks(),
        ConsolidateBlocks(basis_gates=basis_gates),
        UnitarySynthesis(basis_gates),
    ]
)

Para ipakita ang pass manager na ito sa aksyon, subukan ito sa isang two-qubit circuit na binubuo ng isang Hadamard na sinusundan ng dalawang magkadikit na CX gate:

In [4]:
from qiskit import QuantumRegister, QuantumCircuit

qubits = QuantumRegister(2, name="q")
circuit = QuantumCircuit(qubits)

a, b = qubits
circuit.h(a)
circuit.cx(a, b)
circuit.cx(b, a)

circuit.draw("mpl")

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/bc208935-e63c-461b-90d0-a6f4cabc16b6-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/bc208935-e63c-461b-90d0-a6f4cabc16b6-0.svg)

Para patakbuhin ang pass manager sa circuit, tawagan ang `run` method.

In [5]:
translated = translate.run(circuit)
translated.draw("mpl")

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/adb5c242-5cba-4878-a00d-4ec47737d029-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/adb5c242-5cba-4878-a00d-4ec47737d029-0.svg)

Para sa isang mas advanced na halimbawa na nagpapakita kung paano gumawa ng pass manager para ipatupad ang error suppression technique na kilala bilang dynamical decoupling, tingnan ang [Create a pass manager for dynamical decoupling](dynamical-decoupling-pass-manager).

## Gumawa ng staged pass manager

Ang `StagedPassManager` ay isang pass manager na binubuo ng mga indibidwal na yugto, kung saan ang bawat yugto ay tinukoy ng isang `PassManager` instance. Maaari kang lumikha ng `StagedPassManager` sa pamamagitan ng pagtukoy ng mga nais na yugto. Halimbawa, ang sumusunod na code ay gumagawa ng staged pass manager na may dalawang yugto, `init` at `translation`. Ang `translation` na yugto ay tinukoy ng pass manager na ginawa kanina.

In [6]:
from qiskit.transpiler import PassManager, StagedPassManager
from qiskit.transpiler.passes import UnitarySynthesis, Unroll3qOrMore

basis_gates = ["rx", "ry", "rxx"]
init = PassManager(
    [UnitarySynthesis(basis_gates, min_qubits=3), Unroll3qOrMore()]
)
staged_pm = StagedPassManager(
    stages=["init", "translation"], init=init, translation=translate
)

Walang limitasyon sa bilang ng mga yugto na maaari mong ilagay sa isang staged pass manager.

Ang isa pang kapaki-pakinabang na paraan para gumawa ng staged pass manager ay ang magsimula sa isang preset staged pass manager at pagkatapos ay palitan ang ilan sa mga yugto. Halimbawa, ang sumusunod na code ay gumagawa ng preset pass manager na may optimization level 3, at pagkatapos ay nagtutukoy ng custom na `pre_layout` na yugto.

In [7]:
import numpy as np
from qiskit.circuit.library import HGate, PhaseGate, RXGate, TdgGate, TGate
from qiskit.transpiler.passes import InverseCancellation

pass_manager = generate_preset_pass_manager(3, backend)
inverse_gate_list = [
    HGate(),
    (RXGate(np.pi / 4), RXGate(-np.pi / 4)),
    (PhaseGate(np.pi / 4), PhaseGate(-np.pi / 4)),
    (TGate(), TdgGate()),
]
logical_opt = PassManager(
    [
        InverseCancellation(inverse_gate_list),
    ]
)

# Add pre-layout stage to run extra logical optimization
pass_manager.pre_layout = logical_opt

Ang mga [stage generator function](https://docs.quantum.ibm.com/api/qiskit/transpiler_preset#stage-generator-functions) ay maaaring maging kapaki-pakinabang sa pagbuo ng mga custom na pass manager.
Gumagawa ang mga ito ng mga yugto na nagbibigay ng karaniwang functionality na ginagamit sa maraming pass manager.
Halimbawa, ang [`generate_embed_passmanager`](https://docs.quantum.ibm.com/api/qiskit/transpiler_preset#qiskit.transpiler.preset_passmanagers.generate_embed_passmanager) ay maaaring gamitin para gumawa ng yugto
para "i-embed" ang isang piling paunang `Layout` mula sa isang layout pass patungo sa tinukoy na target na device.

## Mga susunod na hakbang
> **Tip:** - [Gumawa ng custom na transpiler pass](custom-transpiler-pass).
>     - [Gumawa ng pass manager para sa dynamical decoupling](dynamical-decoupling-pass-manager).
>     - Para matuto pa tungkol sa `generate_preset_passmanager` function, tingnan ang paksa na [Transpilation default settings and configuration options](defaults-and-configuration-options).
>     - Subukan ang gabay na [Compare transpiler settings](/guides/circuit-transpilation-settings).
>     - Suriin ang [dokumentasyon ng transpiler API.](https://docs.quantum.ibm.com/api/qiskit/transpiler)